# 02 - Apply Finance Adjustments
Owner: Finance Systems (Elena Brandt). Applies manual credit notes / accrual adjustments to silver orders.
Table names are driven by the job config so the same notebook runs in dev, test and prod.

In [ ]:
from pyspark.sql import functions as F

CONFIG = {
    "orders_table":      "harborline.silver.orders_clean",
    "adjustments_table": "harborline.finance.revenue_adjustments",
    "output_table":      "harborline.silver.orders_adjusted",
}


In [ ]:
orders = spark.table(CONFIG["orders_table"])
adjustments = spark.table(CONFIG["adjustments_table"])

adjusted = (
    orders.join(adjustments, "order_id", "left")
          .withColumn(
              "adjusted_net_amount",
              F.col("net_amount") + F.coalesce(F.col("adjustment_amount"), F.lit(0)),
          )
)
adjusted.write.mode("overwrite").saveAsTable(CONFIG["output_table"])
